# 01c · SMLP4Rec với query token (nhận biết ngữ cảnh, không hybrid)

Notebook 01 chỉ đưa cho SMLP4Rec các hotel cluster trước đó của user. Ở đây **lượt tìm kiếm hiện tại** đi vào mô hình như một vị trí bổ sung (query token). Kế hoạch: `docs/plans/plan_oct_07_query_token.md`.

- **Lịch sử**: cùng chuỗi 20 cluster như nb01 (RecBole right-padding), vị trí 0..19.
- **Query token** (vị trí 20): `[MASK]` học được + tổng của một embedding cho mỗi trường query trong lượt tìm kiếm của chính booking mục tiêu: destination (từ vựng train, số lần >= 5), loại destination, tháng check-in, bucket lead-time, bucket độ dài lưu trú, người lớn, trẻ em, phòng, package, mobile, channel, site, châu lục point-of-sale, quốc gia user (số lần >= 5). Hidden state ở vị trí 20 chấm điểm 100 cluster.
- **Không dùng**: feature ẩn của `destinations.csv`, các trường khách sạn (`hotel_*`), `orig_destination_distance`, `user_location_region/city`, `cnt`.
- **Giống nb01**: cách chia (thời gian toàn cục 80/10/10 trên các mục tiêu next-booking, chỉ các dòng warm), config (2 lớp, hidden 64, dropout 0, CE, Adam lr 1e-3, batch 1024, 3 epoch, seed 2022), xếp hạng đầy đủ trên 100 cluster, Recall@K / NDCG@K với K = 5, 10, 20 (`nbp.eval.metrics`).
- **Không bao gồm**: prior, sameDest, hybrid, ngữ cảnh theo từng vị trí, user cold (L = 0). Một seed; 3 epoch là ngân sách kiểm chứng pipeline, không phải kết quả đã tinh chỉnh.
- **Kiểm tra**: mô hình plain (nb01 `best.pth`) được chấm lại đúng trên các dòng này và phải tái tạo số test của nb01, chứng minh các dòng là như nhau. Chẩn đoán xáo trộn query và che lịch sử cho thấy mô hình dựa vào gì.

Dữ liệu: `scripts/expedia_query_to_recbole.py` ghi `expedia_rowid.inter` (cùng các dòng với `expedia.inter` cộng `row_id`) và `expedia_query_codes.npz`. Kernel: `smlp4rec`.

In [1]:
import functools
import json
import logging
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # import torch before pandas: on Windows, pandas first makes c10.dll fail to load

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)  # RecBole checkpoints store the Config

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.data.query_features import FORBIDDEN_EXACT, FORBIDDEN_PREFIX, QUERY_FIELDS, RAW_COLS, build_vocab, encode
from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import ndcg_at_k, summarize, target_rank
from nbp.models.smlprec_query import ROW_ID, SMLPRECQuery
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)
logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
CODES = ROOT / "data" / "interim" / "expedia_query_codes.npz"
PLAIN_RUN = ROOT / "experiments" / "2026-10-05_expedia_smlp4rec_week3-repro"
PLAIN_JSON = ROOT / "results" / "week3_implementation" / "smlprec_expedia_run.json"
RESULT_JSON = ROOT / "results" / "week4_rebuild" / "smlprec_query_run.json"
TAG = "query-1c"
RUN_DIR = ROOT / "experiments" / f"{date.today():%Y-%m-%d}_expedia_smlp4rec_{TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESUME = False  # True: continue from RUN_DIR/last.pth (weights + optimizer + per-epoch rows)
KS = (5, 10, 20)
print("run dir:", RUN_DIR)

# leak guard: no forbidden raw column feeds the query features
assert not [c for c in RAW_COLS if c in FORBIDDEN_EXACT or c.startswith(FORBIDDEN_PREFIX)]
print("query fields:", list(QUERY_FIELDS))

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-07_expedia_smlp4rec_query-1c
query fields: ['dest', 'dest_type', 'ci_month', 'lead', 'stay', 'adults', 'children', 'rooms', 'package', 'mobile', 'channel', 'site', 'posa', 'country']


## 1. Config và dữ liệu

Cùng config RecBole và seed như nb01; chỉ khác dataset (`expedia_rowid` = `expedia.inter` + `row_id`). RecBole sao chép `row_id` sang mọi dòng mục tiêu, nhờ đó mô hình tra được feature query của mục tiêu.

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_rowid",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "row_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])

t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
t_data = time.time() - t0

ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
IID, TIME, UID = config["ITEM_ID_FIELD"], config["TIME_FIELD"], config["USER_ID_FIELD"]


def split_inputs(loader) -> dict:
    # history ids (n, 20), history length (n,), target row id (n,), true next cluster (n,), user (n,)
    d = loader.dataset.inter_feat
    return {
        "seq": d[ITEM_SEQ],
        "length": d[ITEM_SEQ_LEN],
        "row": d[ROW_ID].long(),
        "y": d[IID].numpy(),
        "user": d[UID].numpy(),
    }


eval_inputs = {"valid": split_inputs(valid_data), "test": split_inputs(test_data)}
splits = {}
for name, loader in (("train", train_data), ("valid", valid_data), ("test", test_data)):
    d = loader.dataset.inter_feat
    ts = d[TIME].numpy()
    splits[name] = {
        "targets": len(ts),
        "first_target_unix": int(ts.min()),
        "last_target_unix": int(ts.max()),
        "mean_history_len": round(float(d[ITEM_SEQ_LEN].float().mean()), 3),
    }
print(f"data build {t_data:.1f}s; users {dataset.user_num - 1}, clusters {dataset.item_num - 1}")
print(json.dumps(splits, indent=1))

plain_ref = json.loads(PLAIN_JSON.read_text(encoding="utf-8"))
for name in ("train", "valid", "test"):
    for k in ("targets", "first_target_unix", "last_target_unix", "mean_history_len"):
        assert splits[name][k] == plain_ref["splits"][name][k], (name, k)
print("split sizes, time ranges and mean history length identical to nb01")

data build 66.6s; users 813985, clusters 100
{
 "train": {
  "targets": 1749368,
  "first_target_unix": 1357518976,
  "last_target_unix": 1412776704,
  "mean_history_len": 5.853
 },
 "valid": {
  "targets": 218670,
  "first_target_unix": 1412776704,
  "last_target_unix": 1416398080,
  "mean_history_len": 7.391
 },
 "test": {
  "targets": 218670,
  "first_target_unix": 1416398080,
  "last_target_unix": 1420070400,
  "mean_history_len": 7.255
 }
}
split sizes, time ranges and mean history length identical to nb01


## 2. Feature query: từ vựng chỉ từ train và độ phủ

`codes[i]` chứa các mã query của booking có `row_id = i`. Một mã chỉ có dòng embedding riêng nếu xuất hiện đủ thường xuyên trong **các mục tiêu train** (destination và quốc gia user: >= 5; các trường khác: >= 1); mọi giá trị còn lại (kể cả giá trị chỉ xuất hiện ở valid/test) ánh xạ về chỉ số 0. Độ phủ = tỷ lệ mục tiêu valid/test có giá trị nằm ngoài từ vựng.

In [3]:
import pandas as pd  # after torch

z = np.load(CODES)
codes, fields = z["codes"], list(z["fields"])
assert fields == list(QUERY_FIELDS)
# one code row per booking of the .inter file (row_id = line index)
assert codes.shape[0] == len(pd.read_csv(WORK / "expedia_rowid" / "expedia_rowid.inter", sep="	", usecols=["row_id:float"])), codes.shape

train_rows = train_data.dataset.inter_feat[ROW_ID].long().numpy()
vocab = build_vocab(codes, train_rows)
Q = encode(codes, vocab)  # (n_bookings, F) int32, 0 = out of vocabulary
field_sizes = [len(v) + 1 for v in vocab]

coverage = pd.DataFrame(
    {
        "vocab_size": [len(v) for v in vocab],
        "oov_valid_targets": [float((Q[eval_inputs["valid"]["row"].numpy(), j] == 0).mean()) for j in range(len(fields))],
        "oov_test_targets": [float((Q[eval_inputs["test"]["row"].numpy(), j] == 0).mean()) for j in range(len(fields))],
    },
    index=fields,
).round(4)
print(coverage)
np.savez_compressed(RUN_DIR / "vocab.npz", fields=np.array(fields), **{f"vocab_{f}": v for f, v in zip(fields, vocab)})

           vocab_size  oov_valid_targets  oov_test_targets
dest            13766             0.0227            0.0243
dest_type           8             0.0000            0.0000
ci_month           12             0.0000            0.0000
lead               11             0.0000            0.0000
stay               14             0.0000            0.0000
adults             10             0.0000            0.0000
children           10             0.0000            0.0000
rooms               9             0.0000            0.0000
package             2             0.0000            0.0000
mobile              2             0.0000            0.0000
channel            11             0.0000            0.0000
site               39             0.0003            0.0003
posa                5             0.0000            0.0000
country           212             0.0000            0.0000


## 3. Mô hình và optimizer

`SMLPRECQuery` (`src/nbp/models/smlprec_query.py`): khối mixer cũ và embedding item cộng một bảng embedding cho mỗi trường query. Optimizer như nb01: Adam, lr 1e-3, không weight decay, không cắt gradient.

In [4]:
Qt = torch.from_numpy(Q)
model = SMLPRECQuery(config, train_data.dataset, Qt, field_sizes).to(config["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
n_params = sum(p.numel() for p in model.parameters())
n_query_params = sum(p.numel() for n, p in model.named_parameters() if n.startswith("query_"))
print(model)
print(f"parameters: {n_params} (query token tables: {n_query_params}; history-only model nb01: {plain_ref['n_parameters']})")

SMLPRECQuery(
  (item_embedding): Embedding(101, 64, padding_idx=0)
  (layers): ModuleList(
    (0): SMLP(
      (c): Sequential(
        (0): Linear(in_features=1, out_features=3, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=3, out_features=1, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (h): Sequential(
        (0): Linear(in_features=21, out_features=63, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=63, out_features=21, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (w): Sequential(
        (0): Linear(in_features=64, out_features=192, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.0, inplace=False)
        (3): Linear(in_features=192, out_features=64, bias=False)
        (4): Dropout(p=0.0, inplace=False)
      )
      (norm2): LayerNorm((64,), eps=1e-0

## 4. Hàm huấn luyện, chấm điểm và đánh giá

In [5]:
def train_epoch(model, loader, optimizer) -> float:
    # one epoch over the train loader; returns the sum of batch losses (as nb01 / RecBole)
    model.train()
    total = 0.0
    for inter in loader:
        inter = inter.to(config["device"])
        optimizer.zero_grad()
        loss = model.calculate_loss(inter)
        if torch.isnan(loss):
            raise ValueError("training loss is nan")
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total


@torch.no_grad()
def full_scores(model, inp: dict, q_idx=None, seq=None, batch: int = 8192) -> np.ndarray:
    # (n, n_items) scores of one split; padding column = -inf. q_idx / seq override the real inputs (diagnostics)
    model.eval()
    seq = inp["seq"] if seq is None else seq
    q_idx = Qt[inp["row"]] if q_idx is None else q_idx
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = model.scores(seq[i : i + batch], q_idx[i : i + batch].long())
        s[:, 0] = float("-inf")
        out.append(s.cpu().numpy())
    return np.concatenate(out)


def evaluate(model, inp: dict, **kw) -> tuple[dict, np.ndarray]:
    rank = target_rank(full_scores(model, inp, **kw), inp["y"])
    return summarize(rank, ks=KS), rank

## 5. Huấn luyện (3 epoch)

Valid và test được chấm sau mỗi epoch (test chỉ để ghi lại); epoch tốt nhất = **NDCG@10 trên valid** cao nhất, cải thiện nghiêm ngặt, như nb01. Checkpoint trong thư mục chạy: `epoch{n}.pth` (trọng số, mỗi epoch), `last.pth` (trọng số + optimizer + các dòng theo epoch, để tiếp tục với `RESUME = True`), và `best.pth` sau khi huấn luyện.

In [6]:
per_epoch, best, start = [], None, 1
if RESUME and (RUN_DIR / "last.pth").exists():
    ck = torch.load(RUN_DIR / "last.pth")
    model.load_state_dict(ck["state_dict"])
    optimizer.load_state_dict(ck["optimizer"])
    per_epoch, best, start = ck["per_epoch"], ck["best"], ck["epoch"] + 1
    print("resumed after epoch", ck["epoch"])

t1 = time.time()
for epoch in range(start, config["epochs"] + 1):
    t = time.time()
    loss = train_epoch(model, train_data, optimizer)
    row = {"epoch": epoch, "train_loss": round(loss, 4), "train_seconds": round(time.time() - t, 1)}
    t = time.time()
    ranks = {}
    for name, inp in eval_inputs.items():
        row[name], ranks[name] = evaluate(model, inp)
    row["eval_seconds"] = round(time.time() - t, 1)
    valid_score = ndcg_at_k(ranks["valid"], 10)
    torch.save(model.state_dict(), RUN_DIR / f"epoch{epoch}.pth")
    if best is None or valid_score > best["valid_score"]:
        best = {"epoch": epoch, "valid_score": valid_score}
    per_epoch.append(row)
    torch.save(
        {"epoch": epoch, "state_dict": model.state_dict(), "optimizer": optimizer.state_dict(),
         "per_epoch": per_epoch, "best": best},
        RUN_DIR / "last.pth",
    )
    print(
        f"epoch {epoch}: loss {row['train_loss']:.4f} (train {row['train_seconds']}s, eval {row['eval_seconds']}s) | "
        f"valid R@5 {row['valid']['recall@5']:.4f} NDCG@10 {row['valid']['ndcg@10']:.4f} | "
        f"test R@5 {row['test']['recall@5']:.4f} NDCG@10 {row['test']['ndcg@10']:.4f}"
    )
t_fit = time.time() - t1
print(f"train + per-epoch eval {t_fit:.1f}s; best epoch by valid NDCG@10: {best['epoch']}")

epoch 1: loss 5183.8896 (train 198.7s, eval 15.2s) | valid R@5 0.5832 NDCG@10 0.4716 | test R@5 0.5731 NDCG@10 0.4647


epoch 2: loss 4631.6362 (train 223.4s, eval 14.7s) | valid R@5 0.5965 NDCG@10 0.4828 | test R@5 0.5880 NDCG@10 0.4764


epoch 3: loss 4522.5900 (train 253.6s, eval 23.3s) | valid R@5 0.5981 NDCG@10 0.4832 | test R@5 0.5898 NDCG@10 0.4772
train + per-epoch eval 729.1s; best epoch by valid NDCG@10: 3


## 6. Epoch tốt nhất: lưu `best.pth`, chấm valid và test một lần

`best.pth` chứa trọng số cùng những gì cần để dựng lại mô hình mà không cần notebook này: `field_sizes`, `fields`, và các khóa config định nghĩa kiến trúc. `vocab.npz` (cùng thư mục) ánh xạ mã thô sang chỉ số embedding; tra cứu `row_id` cần `expedia_query_codes.npz`.

In [7]:
state = torch.load(RUN_DIR / f"epoch{best['epoch']}.pth")
model.load_state_dict(state)
ARCH_KEYS = ("MAX_ITEM_LIST_LENGTH", "n_layers", "hidden_size", "hidden_dropout_prob", "hidden_act",
             "layer_norm_eps", "initializer_range", "loss_type", "seed")
torch.save(
    {"epoch": best["epoch"], "state_dict": state, "field_sizes": field_sizes, "fields": fields,
     "config": {k: config[k] for k in ARCH_KEYS}},
    RUN_DIR / "best.pth",
)
t2 = time.time()
valid_best, rank_valid_q = evaluate(model, eval_inputs["valid"])
test_best, rank_test_q = evaluate(model, eval_inputs["test"])
t_test = time.time() - t2
print("valid:", valid_best)
print("test: ", test_best)

valid: {'n': 218670, 'recall@5': 0.5981, 'recall@10': 0.7725, 'recall@20': 0.9094, 'ndcg@5': 0.4266, 'ndcg@10': 0.4832, 'ndcg@20': 0.5181}
test:  {'n': 218670, 'recall@5': 0.5898, 'recall@10': 0.7636, 'recall@20': 0.9029, 'ndcg@5': 0.4209, 'ndcg@10': 0.4772, 'ndcg@20': 0.5127}


## 7. Các dòng tham chiếu trên cùng dòng test: mô hình plain, popularity, repeat-last

Mô hình plain (chỉ lịch sử) là checkpoint tốt nhất của nb01 được chấm lại trên **các** dòng này. Số test của nó phải bằng của nb01 (`results/week3_implementation/smlprec_expedia_run.json`); nếu đúng, các dòng, lịch sử và mục tiêu giống hệt nb01.

In [8]:
plain = SMLPREC(config, train_data.dataset).to(config["device"])
plain.load_state_dict(torch.load(PLAIN_RUN / "best.pth")["state_dict"])
plain.eval()


@torch.no_grad()
def plain_rank(inp: dict, batch: int = 8192) -> np.ndarray:
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = plain.full_sort_predict({plain.ITEM_SEQ: inp["seq"][i : i + batch], plain.ITEM_SEQ_LEN: inp["length"][i : i + batch]})
        s[:, 0] = float("-inf")
        out.append(s.numpy())
    return target_rank(np.concatenate(out), inp["y"])


rank_valid_p, rank_test_p = plain_rank(eval_inputs["valid"]), plain_rank(eval_inputs["test"])
plain_test, plain_valid = summarize(rank_test_p, ks=KS), summarize(rank_valid_p, ks=KS)
for k, v in plain_ref["test"].items():
    if k in plain_test:
        assert abs(plain_test[k] - v) < 1e-4, (k, plain_test[k], v)
print("plain rescored on these rows reproduces nb01 test:", {k: plain_test[k] for k in ("recall@5", "ndcg@5", "ndcg@10")})

n_items = dataset.item_num
pop = np.bincount(train_data.dataset.inter_feat[IID].numpy(), minlength=n_items).astype(np.float64)
pop[0] = -np.inf
y = eval_inputs["test"]["y"]
seq_np, len_np = eval_inputs["test"]["seq"].numpy(), eval_inputs["test"]["length"].numpy()
last = seq_np[np.arange(len(y)), len_np - 1]
s_pop = np.tile(pop, (len(y), 1))
s_rep = s_pop.copy()
s_rep[np.arange(len(y)), last] = pop.max() + 1
heuristics = {
    "global_popularity": summarize(target_rank(s_pop, y), ks=KS),
    "repeat_last_then_popularity": summarize(target_rank(s_rep, y), ks=KS),
}
del s_pop, s_rep

plain rescored on these rows reproduces nb01 test: {'recall@5': 0.3369, 'ndcg@5': 0.2439, 'ndcg@10': 0.285}


## 8. Kết quả (test, các dòng warm L >= 1, xếp hạng đầy đủ trên 100 cluster)

In [9]:
COLS = ["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
table = pd.DataFrame(
    {
        "global popularity": heuristics["global_popularity"],
        "repeat-last + popularity": heuristics["repeat_last_then_popularity"],
        "SMLP4Rec plain (history only, nb01)": plain_test,
        "SMLP4Rec + query token (1c)": test_best,
    }
).T[["n", *COLS]]
print(f"TEST (best epoch {best['epoch']} by valid NDCG@10)")
table

TEST (best epoch 3 by valid NDCG@10)


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
global popularity,218670.0,0.1478,0.0938,0.2476,0.1256,0.3925,0.1619
repeat-last + popularity,218670.0,0.2570,0.2031,0.3439,0.2309,0.4762,0.2641
"SMLP4Rec plain (history only, nb01)",218670.0,0.3369,0.2439,0.4645,0.2850,0.6309,0.3269
SMLP4Rec + query token (1c),218670.0,0.5898,0.4209,0.7636,0.4772,0.9029,0.5127


### Theo việc user có trong train hay không, và theo độ dài lịch sử

*Seen* = user có ít nhất một mục tiêu train (như nb06). *Unseen, L >= 1* = user có booking trước đó nhưng không có mục tiêu train. User cold (L = 0) không nằm trong các dòng này.

In [10]:
train_users = np.unique(train_data.dataset.inter_feat[UID].numpy())
seen = np.isin(eval_inputs["test"]["user"], train_users)
L = eval_inputs["test"]["length"].numpy()
masks = {
    "all warm rows": np.ones(len(L), bool),
    "seen users": seen,
    "unseen users (L>=1)": ~seen,
    "L=1": L == 1,
    "L=2-4": (L >= 2) & (L <= 4),
    "L=5-9": (L >= 5) & (L <= 9),
    "L>=10": L >= 10,
}
rows = []
for sl, m in masks.items():
    for name, r in (("plain", rank_test_p), ("query token", rank_test_q)):
        s = summarize(r[m], ks=(5,))
        rows.append({"slice": sl, "model": name, "n": s["n"], "recall@5": s["recall@5"], "ndcg@5": s["ndcg@5"]})
slices = pd.DataFrame(rows).set_index(["slice", "model"])
print(slices)

boot = {sl: paired_bootstrap(rank_test_q[m], rank_test_p[m], k=5) for sl, m in masks.items() if sl in ("all warm rows", "seen users", "unseen users (L>=1)")}
print("\nTEST, query token minus plain, paired bootstrap (1000 resamples, seed 0, 95% CI)")
pd.DataFrame(
    {sl: {m: f"{v[m]['diff']:+.4f} [{v[m]['ci95'][0]:+.4f}, {v[m]['ci95'][1]:+.4f}]" for m in ("recall@5", "ndcg@5")} for sl, v in boot.items()}
).T

                                      n  recall@5  ndcg@5
slice               model                                
all warm rows       plain        218670    0.3369  0.2439
                    query token  218670    0.5898  0.4209
seen users          plain        143025    0.3406  0.2433
                    query token  143025    0.5997  0.4282
unseen users (L>=1) plain         75645    0.3299  0.2451
                    query token   75645    0.5710  0.4071
L=1                 plain         39993    0.2691  0.1971
                    query token   39993    0.5549  0.3891
L=2-4               plain         68253    0.3155  0.2275
                    query token   68253    0.5771  0.4097
L=5-9               plain         47412    0.3551  0.2556
                    query token   47412    0.5997  0.4293
L>=10               plain         63012    0.3893  0.2826
                    query token   63012    0.6182  0.4467



TEST, query token minus plain, paired bootstrap (1000 resamples, seed 0, 95% CI)


,recall@5,ndcg@5
all warm rows,"+0.2529 [+0.2504, +0.2551]","+0.1770 [+0.1752, +0.1785]"
seen users,"+0.2591 [+0.2561, +0.2621]","+0.1849 [+0.1827, +0.1870]"
unseen users (L>=1),"+0.2412 [+0.2374, +0.2450]","+0.1619 [+0.1593, +0.1647]"


## 9. Chẩn đoán (cùng checkpoint, không huấn luyện, không tinh chỉnh)

- **Xáo trộn query**: mỗi dòng test nhận feature query của một dòng test khác (lịch sử giữ nguyên). Nếu mô hình dùng query, điểm sẽ giảm.
- **Che lịch sử**: lịch sử được thay bằng padding (query giữ nguyên). Cho thấy riêng query mang được gì.
- **Cả hai**: không có thông tin nào về dòng đó.

In [11]:
inp = eval_inputs["test"]
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(inp["y"]), generator=g)
q_real = Qt[inp["row"]]
zeros_seq = torch.zeros_like(inp["seq"])
diag = {
    "real": test_best,
    "query shuffled": evaluate(model, inp, q_idx=q_real[perm])[0],
    "history masked": evaluate(model, inp, seq=zeros_seq)[0],
    "both (shuffled + masked)": evaluate(model, inp, q_idx=q_real[perm], seq=zeros_seq)[0],
}
print("TEST diagnostics, SMLP4Rec + query token")
pd.DataFrame(diag).T[["n", *COLS]]

TEST diagnostics, SMLP4Rec + query token


,n,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
real,218670.0,0.5898,0.4209,0.7636,0.4772,0.9029,0.5127
query shuffled,218670.0,0.1315,0.0864,0.2203,0.1148,0.3699,0.1523
history masked,218670.0,0.5165,0.3496,0.7090,0.4120,0.8741,0.4540
both (shuffled + masked),218670.0,0.0774,0.0462,0.1476,0.0686,0.2772,0.1011


## 10. Lưu `smlprec_query_run.json` và `metrics.json`

Mọi con số được trích dẫn sau này trong slide hay báo cáo đều đến từ file này.

In [12]:
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=False).stdout.strip()
result = {
    "run": "SMLP4Rec + query token (context-aware, no hybrid), 3 epochs, notebook 01c",
    "dataset": "expedia (train.csv, is_booking == 1; item = hotel_cluster)",
    "git_commit": commit,
    "query_fields": fields,
    "query_vocab_min_count": dict(QUERY_FIELDS),
    "excluded_inputs": "destinations.csv latent features, hotel_*, orig_destination_distance, user_location_region/city, cnt",
    "config": {k: config[k] for k in ("MAX_ITEM_LIST_LENGTH", "n_layers", "hidden_size", "hidden_dropout_prob",
                                      "loss_type", "epochs", "train_batch_size", "learning_rate", "seed", "eval_args", "topk", "valid_metric")},
    "n_parameters": int(n_params),
    "n_query_parameters": int(n_query_params),
    "n_parameters_history_only": plain_ref["n_parameters"],
    "splits": splits,
    "query_coverage": coverage.to_dict(orient="index"),
    "timing_seconds": {"data_build": round(t_data, 1), "train_and_eval": round(t_fit, 1), "test": round(t_test, 1)},
    "best_epoch_by_valid": best["epoch"],
    "checkpoint": str(RUN_DIR / "best.pth"),
    "per_epoch": per_epoch,
    "valid": valid_best,
    "test": test_best,
    "plain_rescored_valid": plain_valid,
    "plain_rescored_test": plain_test,
    "heuristics_test": heuristics,
    "slices_test": {f"{a} | {b}": v for (a, b), v in slices.to_dict(orient="index").items()},
    "bootstrap_query_minus_plain_test": boot,
    "diagnostics_test": diag,
}
RESULT_JSON.parent.mkdir(parents=True, exist_ok=True)
for path in (RESULT_JSON, RUN_DIR / "metrics.json"):
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("wrote", RESULT_JSON, "and", RUN_DIR / "metrics.json")

wrote C:\workspace\PROJECTS\next-best-product-recommendation\results\week4_rebuild\smlprec_query_run.json and C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-07_expedia_smlp4rec_query-1c\metrics.json
